# Skin Lesion Classification — PAD-UFES-20
**Week 2 deliverable:** annotated dataset + classification code + metrics.

Project 1 (Patient Skin Lesion Monitoring), Computer Vision Semester 6.  
Model: switchable **MobileNetV2 / ResNet50** (transfer learning, ImageNet).

Improvements over the baseline: balanced batch sampling, stronger
augmentation, two-stage fine-tuning (freeze -> unfreeze), cosine LR
schedule, and best-model selection by **macro-F1** (handles class imbalance).

## How to use
1. Runtime -> Change runtime type -> **GPU (T4)** -> Save.
2. In Drive folder `skin-lesion-monitoring-cv_dataset` put `images.zip`
   and `metadata.csv`.
3. Run top to bottom. Edit only the Config cell (`DATASET_DIR` / `MODEL_NAME`).
4. Compare models: run with `MODEL_NAME='mobilenet_v2'`, then `'resnet50'`.

In [ ]:
# 1. Imports + GPU check
import os, glob, zipfile, time, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import models, transforms
from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)
if device.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: no GPU. Runtime -> Change runtime type -> GPU (T4).')

In [ ]:
# 2. Mount Google Drive (force_remount fixes 'Transport endpoint' errors)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# 3. Config -- edit DATASET_DIR and MODEL_NAME as needed
DATASET_DIR = '/content/drive/MyDrive/skin-lesion-monitoring-cv_dataset'
ZIP_ON_DRIVE  = os.path.join(DATASET_DIR, 'images.zip')
META_ON_DRIVE = os.path.join(DATASET_DIR, 'metadata.csv')
DATA_ROOT = '/content/pad-ufes-20'               # zip extracted here (fast local disk)
OUT_DIR   = '/content/drive/MyDrive/pad-ufes-20-results'
  
MODEL_NAME   = 'mobilenet_v2'   # 'resnet50' or 'mobilenet_v2'  
IMG_SIZE     = 224  
BATCH_SIZE   = 32
EPOCHS       = 25              # total epochs (was 10)
FREEZE_EPOCHS = 3             # epochs training only the head before unfreezing
HEAD_LR      = 1e-3           # LR while backbone is frozen
FINE_LR      = 1e-4           # LR after unfreezing (fine-tune whole net)
VAL_SPLIT    = 0.2
SEED         = 42
CLASSES      = ['BCC', 'SCC', 'ACK', 'SEK', 'MEL', 'NEV']

torch.manual_seed(SEED); np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.isfile(ZIP_ON_DRIVE),  f'Zip not found: {ZIP_ON_DRIVE}'
assert os.path.isfile(META_ON_DRIVE), f'metadata.csv not found: {META_ON_DRIVE}'
print('Zip OK :', ZIP_ON_DRIVE)
print('Meta OK:', META_ON_DRIVE)
print('Model  :', MODEL_NAME)

In [ ]:
# 4. Copy zip + metadata to LOCAL disk first (robust), then unzip locally.
local_zip  = '/content/images.zip'
local_meta = '/content/metadata.csv'

t = time.time()
shutil.copy(ZIP_ON_DRIVE, local_zip)
shutil.copy(META_ON_DRIVE, local_meta)
print(f'Copied zip+csv to local disk in {time.time() - t:.0f}s '
      f'({os.path.getsize(local_zip) / 1e6:.0f} MB zip)')

t = time.time()
os.makedirs(DATA_ROOT, exist_ok=True)
with zipfile.ZipFile(local_zip) as z:
    z.extractall(DATA_ROOT)
shutil.copy(local_meta, os.path.join(DATA_ROOT, 'metadata.csv'))
print(f'Extracted in {time.time() - t:.0f}s into {DATA_ROOT}')

n_img = sum(len(glob.glob(os.path.join(DATA_ROOT, '**', e), recursive=True))
            for e in ('*.png', '*.jpg', '*.jpeg'))
print('Images found after extract:', n_img)

In [ ]:
# 5. Build the annotated dataset (image path -> class label)
paths = {}
for ext in ('*.png', '*.jpg', '*.jpeg'):
    for p in glob.glob(os.path.join(DATA_ROOT, '**', ext), recursive=True):
        paths[os.path.basename(p)] = p
print('Indexed images:', len(paths))

meta = pd.read_csv(os.path.join(DATA_ROOT, 'metadata.csv'))
meta = meta[['img_id', 'diagnostic']].dropna()
meta = meta[meta['diagnostic'].isin(CLASSES)].copy()
meta['path'] = meta['img_id'].map(paths)
meta = meta.dropna(subset=['path']).reset_index(drop=True)
meta['label'] = meta['diagnostic'].map({c: i for i, c in enumerate(CLASSES)})

print('Usable samples:', len(meta))
print(meta['diagnostic'].value_counts())
meta.to_csv(os.path.join(OUT_DIR, 'annotated_dataset.csv'), index=False)

In [ ]:
# 6. Stronger augmentation + BALANCED batch sampling (fixes class imbalance)
train_df, val_df = train_test_split(
    meta, test_size=VAL_SPLIT, stratify=meta['label'], random_state=SEED)
print('Train:', len(train_df), ' Val:', len(val_df))

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
# (2) Stronger augmentation: random resized crop, flips, rotation, color jitter
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2,
                           saturation=0.2, hue=0.02),
    transforms.ToTensor(), norm])
val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(), norm])

class LesionDS(Dataset):
    def __init__(self, df, tf):
        self.df = df.reset_index(drop=True); self.tf = tf
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(r['path']).convert('RGB')
        return self.tf(img), int(r['label'])

# (1) Balanced sampler: rare classes (SCC, MEL) drawn more often per batch
cls_count = train_df['label'].value_counts().sort_index().values
cls_w = 1.0 / cls_count
samp_w = train_df['label'].map(lambda c: cls_w[c]).values
sampler = WeightedRandomSampler(torch.DoubleTensor(samp_w),
                                num_samples=len(samp_w), replacement=True)

train_dl = DataLoader(LesionDS(train_df, train_tf), batch_size=BATCH_SIZE,
                      sampler=sampler, num_workers=2)
val_dl = DataLoader(LesionDS(val_df, val_tf), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=2)

In [ ]:
# 7. Model (switchable). Optimizer/scheduler built in the training cell
# because of the two-stage freeze -> unfreeze schedule.
def build_model(name):
    if name == 'resnet50':
        m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Linear(m.fc.in_features, len(CLASSES))
        head = m.fc
    elif name == 'mobilenet_v2':
        m = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, len(CLASSES))
        head = m.classifier
    else:
        raise ValueError(f'Unknown MODEL_NAME: {name}')
    return m, head

def set_trainable(model, head, backbone_trainable):
    for p in model.parameters():
        p.requires_grad = backbone_trainable
    for p in head.parameters():
        p.requires_grad = True   # head always trainable

model, head = build_model(MODEL_NAME)
model = model.to(device)
# Balanced sampler already equalises classes, so plain (unweighted) loss.
criterion = nn.CrossEntropyLoss()
print('Model:', MODEL_NAME)

In [ ]:
# 8. Two-stage training + cosine LR + best model chosen by MACRO-F1
# Phase 1 (epochs 1..FREEZE_EPOCHS): backbone frozen, train head only.
# Phase 2 (rest): unfreeze all, low LR, cosine annealing.
set_trainable(model, head, backbone_trainable=False)
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad], lr=HEAD_LR)
scheduler = None
best_f1 = 0.0

for epoch in range(1, EPOCHS + 1):
    if epoch == FREEZE_EPOCHS + 1:                 # (4) unfreeze backbone
        set_trainable(model, head, backbone_trainable=True)
        optimizer = torch.optim.Adam(model.parameters(), lr=FINE_LR)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - FREEZE_EPOCHS)   # (3) cosine schedule
        print(f'-- epoch {epoch}: backbone unfrozen, fine-tuning --')

    model.train(); tr_loss = 0.0
    for x, y in train_dl:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(x), y)
        loss.backward(); optimizer.step()
        tr_loss += loss.item() * x.size(0)
    tr_loss /= len(train_dl.dataset)
    if scheduler is not None:
        scheduler.step()

    model.eval(); yt, yp = [], []
    with torch.no_grad():
        for x, y in val_dl:
            x = x.to(device)
            yp += model(x).argmax(1).cpu().tolist()
            yt += y.tolist()
    val_acc = np.mean(np.array(yt) == np.array(yp))
    macro_f1 = f1_score(yt, yp, average='macro')
    print(f'Epoch {epoch:2d}/{EPOCHS}  loss={tr_loss:.4f}  '
          f'val_acc={val_acc:.4f}  macro_f1={macro_f1:.4f}')
    if macro_f1 > best_f1:                          # (5) select by macro-F1
        best_f1 = macro_f1
        torch.save(model.state_dict(),
                   os.path.join(OUT_DIR, f'best_{MODEL_NAME}.pth'))
print('Best macro-F1:', round(best_f1, 4))

In [ ]:
# 9. Metrics: classification report + confusion matrix (best model)
model.load_state_dict(torch.load(os.path.join(OUT_DIR, f'best_{MODEL_NAME}.pth')))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in val_dl:
        x = x.to(device)
        y_pred += model(x).argmax(1).cpu().tolist()
        y_true += y.tolist()

report = classification_report(y_true, y_pred, target_names=CLASSES, digits=4)
print(report)
with open(os.path.join(OUT_DIR, f'metrics_{MODEL_NAME}.txt'), 'w') as f:
    f.write(f'Model: {MODEL_NAME} (transfer learning, improved pipeline)\n')
    f.write('Best macro-F1: %.4f\n\n' % best_f1)
    f.write(report)

cm = confusion_matrix(y_true, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title(f'Confusion Matrix ({MODEL_NAME})')
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, cm[i, j], ha='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
fig.colorbar(im); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f'confusion_matrix_{MODEL_NAME}.png'), dpi=120)
plt.show()
print('Saved model + metrics + confusion matrix to', OUT_DIR)